# Quick start: Venture value created per dollar at risk: US venture-backed listings, 2014-2026

This notebook loads the tables with pandas, shows the firm, fund and company rankings, drills into one firm and recomputes its headline figure from the core tables. All values are at the performance end date, the close on 2 October 2026.

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
read = lambda p: pd.read_csv(ROOT / p, keep_default_na=False, na_values=[''])
listings, firms, holdings = read('data/core/listings.csv'), read('data/core/firms.csv'), read('data/core/holdings.csv')
vehicles, allocations, generations = read('data/core/fund_vehicles.csv'), read('data/core/fund_allocations.csv'), read('data/core/fund_generations.csv')
firm_metrics, fund_metrics, companies = read('data/derived/firm_metrics.csv'), read('data/derived/fund_metrics.csv'), read('data/derived/listing_venture_summary.csv')
names = firms.set_index('firm_id').firm_name
firm_metrics['rank'] = firm_metrics['rank'].astype('Int64'); fund_metrics['rank'] = fund_metrics['rank'].astype('Int64')
{t: len(df) for t, df in [('listings', listings), ('firms', firms), ('holdings', holdings), ('fund vehicles', vehicles)]}

{'listings': 424, 'firms': 792, 'holdings': 1955, 'fund vehicles': 3422}

## The firm ranking

Headline: consistency-adjusted value per dollar of capital at risk, with the range from estimated inputs.

In [2]:
top = firm_metrics[firm_metrics.measurement_status == 'ranked'].sort_values('rank').head(15)
top.assign(firm=top.firm_id.map(names))[['rank', 'firm', 'consistency_adjusted_value_per_dollar', 'adjusted_low', 'adjusted_high', 'value_per_dollar', 'value_per_dollar_beyond_largest', 'value_primary_usd_bn', 'capital_at_risk_usd_bn']].round(2)

,rank,firm,consistency_adjusted_value_per_dollar,adjusted_low,adjusted_high,value_per_dollar,value_per_dollar_beyond_largest,value_primary_usd_bn,capital_at_risk_usd_bn
0,1,Sutter Hill,11.55,7.85,16.61,14.37,7.37,19.11,1.33
1,2,Founders Fund,11.22,10.57,11.60,15.46,5.73,95.24,6.16
2,3,IA Ventures,9.99,9.06,10.04,12.41,6.39,4.27,0.34
3,4,Gigafund,9.06,6.74,10.39,30.90,0.00,11.04,0.36
4,5,Lowercase Capital,8.80,6.53,12.17,23.40,0.58,2.29,0.10
5,6,Pelion,7.30,6.89,7.53,22.92,0.00,15.14,0.66
6,7,Union Square Ventures,6.26,5.37,7.26,6.98,5.01,10.21,1.46
7,8,Benchmark,6.00,5.85,6.28,7.58,3.65,17.99,2.37
8,9,Bessemer,5.12,4.83,5.27,7.00,2.58,48.40,6.92
9,10,DFJ / Draper Fisher Jurvetson,4.37,2.77,5.44,7.73,1.03,29.78,3.85


## The fund ranking

In [3]:
fund_metrics[fund_metrics['rank'].notna()].sort_values('rank').head(10)[['rank', 'fund_display_name', 'vintage', 'committed_usd_m', 'value_primary_usd_bn', 'value_per_dollar', 'consistency_adjusted_value_per_dollar', 'adjusted_low', 'adjusted_high']].round(2)

,rank,fund_display_name,vintage,committed_usd_m,value_primary_usd_bn,value_per_dollar,consistency_adjusted_value_per_dollar,adjusted_low,adjusted_high
0,1,Founders Fund II,2007,220.00,37.40,269.34,183.46,152.90,189.79
1,2,Bullet Time Ventures (Fund I),2009,4.85,0.44,99.94,53.27,36.33,66.49
2,3,Founders Fund III,2010,250.00,16.15,73.44,52.22,48.69,54.25
3,4,Index Ventures VI,2012,458.80,21.21,50.59,39.70,38.93,40.10
4,5,Lowercase Ventures Fund I,2010,8.50,1.95,243.07,38.06,32.41,45.75
5,6,FirstMark Capital I,2006,220.00,12.60,97.82,37.17,34.89,40.74
6,7,Sequoia Capital XII,2006,481.00,15.38,54.64,35.38,31.43,37.95
7,8,IA Venture Strategies Fund I,2010,50.00,2.20,46.55,33.16,30.23,33.16
8,9,First Round Capital II,2008,125.00,4.22,40.34,29.04,28.26,29.58
9,10,UV Partners IV (Pelion),2007,119.63,13.48,152.46,27.66,27.66,27.66


## The companies that created the most value

In [4]:
companies.merge(listings[['listing_id', 'company', 'listing_date']], on='listing_id').head(10)[['rank_by_value_end', 'company', 'listing_date', 'value_end_usd_bn', 'multiple_since_listing', 'venture_funds_stake_pct', 'largest_venture_holders']].round(1)

,rank_by_value_end,company,listing_date,value_end_usd_bn,multiple_since_listing,venture_funds_stake_pct,largest_venture_holders
0,1,Space Exploration Technologies,2026-06-12,2095.4,1.2,10.9,Founders Fund 3.1%; Vy Capital 2.1%; DFJ / Dra...
1,2,Palantir Technologies,2020-09-30,325.9,19.9,10.9,Founders Fund 8.1%; 8VC 1.6%; Mithril Capital ...
2,3,Alibaba Group Holding,2014-09-19,260.9,1.6,0.0,NaN
3,4,Crowdstrike Holdings Inc,2019-06-12,212.5,31.8,22.8,Accel 18.7%; March Capital 2.7%; IVP 1.3%
4,5,Arista Networks,2014-06-06,210.9,77.2,0.0,NaN
5,6,Uber Technologies Inc,2019-05-10,115.5,1.5,19.2,Benchmark 8.8%; GV 3.9%; Lowercase Capital 2.6%
6,7,Shopify,2015-05-21,112.7,89.1,45.0,Bessemer 27.1%; FirstMark 10.6%; OMERS Venture...
7,8,Cloudflare Inc,2019-09-13,102.4,23.3,50.4,NEA 17.9%; Pelion 16.5%; Venrock 14.2%
8,9,Airbnb Inc,2020-12-10,97.0,2.4,27.7,Sequoia Capital 13.7%; Founders Fund 4.4%; Gre...
9,10,Robinhood Markets Inc,2021-07-29,95.0,3.0,32.5,Index Ventures 10.1%; NEA 8.9%; Ribbit Capital...


## One firm in detail

Its stakes, how its value splits between funds, and its capital at risk by vintage.

In [5]:
firm = 'union-square-ventures'
h = holdings[(holdings.firm_id == firm) & (holdings.holder_type == 'core_venture')].merge(listings[['listing_id', 'company']], on='listing_id')
h.groupby('company')[['stake_pct', 'primary_value_counted_usd_bn']].sum().sort_values('primary_value_counted_usd_bn', ascending=False).round(3)

,stake_pct,primary_value_counted_usd_bn
company,,
Twilio,11.930,2.889
Coinbase Global Inc,6.660,2.543
Cloudflare Inc,1.713,1.754
MongoDB,8.067,1.414
Etsy,13.481,1.033
Duolingo Inc,10.300,0.533
LendingClub,3.814,0.042


In [6]:
a = allocations[allocations.firm_id == firm].merge(generations[['generation_id', 'generation_label', 'vintage']], on='generation_id', how='left')
a.assign(credited_to=a.generation_label.fillna(a.vehicle_outside_fund_files)).groupby(['credited_to'])['primary_value_usd_bn'].sum().sort_values(ascending=False).round(3)

credited_to
USV 2008        4.303
USV 2012        2.742
USV Opp 2010    1.847
USV 2004        0.983
USV Opp 2014    0.334
Name: primary_value_usd_bn, dtype: float64

In [7]:
vehicles[(vehicles.firm_id == firm) & (vehicles.in_scope == 'Y')].groupby('vintage')[['committed_usd_m', 'capital_at_risk_usd_m']].sum().round(1)

,committed_usd_m,capital_at_risk_usd_m
vintage,,
2004,125.0,47.6
2008,156.0,122.9
2010,165.0,106.2
2011,167.4,154.6
2014,348.8,308.5
2016,175.0,118.1
2018,449.0,294.5
2020,419.1,105.0
2022,927.0,197.2


## Recomputing the headline

`(1 + V/K)^(2/3) x (1 + (V - V_top)/K)^(1/3) - 1`, from the holdings and fund vehicles alone.

In [8]:
v = h.groupby('listing_id').primary_value_counted_usd_bn.sum()
K = vehicles[(vehicles.firm_id == firm) & (vehicles.in_scope == 'Y')].capital_at_risk_usd_m.sum() / 1000
eff, depth = v.sum() / K, (v.sum() - v.max()) / K
headline = (1 + eff) ** (2 / 3) * (1 + depth) ** (1 / 3) - 1
published = firm_metrics.set_index('firm_id').loc[firm, 'consistency_adjusted_value_per_dollar']
print(f'V = ${v.sum():.2f}bn, K = ${K:.2f}bn, value per $ = {eff:.2f}x, headline = {headline:.3f}x (published {published:.3f}x)')

V = $10.21bn, K = $1.46bn, value per $ = 6.98x, headline = 6.261x (published 6.261x)
